# 05 — `classification` late-information ablation

This notebook is an **additional diagnostic experiment** after the main 01–04 rebuild.

It answers one narrow question:

> **What changes if `classification` is added to the exact selected model architecture?**

The comparison deliberately keeps the current selected-model inputs intact.

### Scenario A — current selected architecture

The required fields are read directly from `artifacts/final_bundle.joblib`.  
In the frozen 03/04 run, the saved contract is:

- `subject`
- `origin`
- `category`
- `type`
- `notifying_country`
- `hazards`

### Scenario B — same architecture + one field

- all Scenario A fields
- **+ `classification`**

`classification` is therefore the **only intended input difference**.

This notebook does **not** replace the selected model from Notebook 03 and does not change the Streamlit app in Notebook 04. It is a diagnostic comparison for understanding why an earlier project version achieved higher scores when later-stage information was included.

## 1. Load the existing rebuild exactly as 03/04 do

This notebook does not recreate TF-IDF parameters or Logistic Regression settings by hand.

Instead, it imports `rasff_core.py`, loads the prepared data, temporal blocks, selected specification and final saved bundle, and then **clones the frozen pipeline architecture**.

That prevents this notebook from silently drifting away from the current rebuild.

In [1]:
from pathlib import Path
import sys
import json
import warnings

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    log_loss,
)
from sklearn.preprocessing import OneHotEncoder

from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "rasff_core.py").exists():
    raise FileNotFoundError(
        "Open this notebook from the extracted RASFF_Rebuild folder, "
        "the same way as notebooks 03 and 04."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from rasff_core import *

df = load_prepared()
blocks = temporal_blocks(df)

bundle_path = ARTIFACTS / "final_bundle.joblib"
spec_path = ARTIFACTS / "selected_spec.json"

if not bundle_path.exists():
    raise FileNotFoundError("Run notebooks 01–03 first: final_bundle.joblib is missing.")

bundle = joblib.load(bundle_path)
selected_spec = json.loads(spec_path.read_text()) if spec_path.exists() else None

print("Prepared rows:", len(df))
print("Temporal blocks:", {k: len(v) for k, v in blocks.items()})
print("Selected spec:", selected_spec)

Prepared rows: 27397
Temporal blocks: {'train': 14668, 'validation': 4748, 'calibration': 3128, 'test': 4853}
Selected spec: {'name': 'raw_lr_C2', 'algorithm': 'lr', 'feature_set': 'raw', 'c': 2.0}


## 2. Verify the real input contract before doing anything

Notebook 03 prints the saved required fields, and Notebook 04 generates its prediction form from the same `bundle["columns"]`.

This cell checks that **05 uses that contract**, rather than a manually reconstructed feature list.

In [2]:
base_columns = list(bundle["columns"])

print("Scenario A — saved model inputs:")
display(pd.Series(base_columns, name="required field"))

expected_from_frozen_03_04 = [
    "subject",
    "origin",
    "category",
    "type",
    "notifying_country",
    "hazards",
]

if base_columns == expected_from_frozen_03_04:
    print("PASS: input contract matches the frozen 03/04 run.")
else:
    warnings.warn(
        "The saved bundle input contract differs from the previously frozen 03/04 output. "
        "This notebook will follow the CURRENT bundle, not the hard-coded reference."
    )

if "classification" in base_columns:
    raise RuntimeError(
        "`classification` is already in the selected bundle. "
        "This is not the intended ablation setup."
    )

if "classification" not in df.columns:
    raise KeyError("Prepared data do not contain `classification`.")

Scenario A — saved model inputs:


0              subject
1               origin
2             category
3                 type
4    notifying_country
5              hazards
Name: required field, dtype: object

PASS: input contract matches the frozen 03/04 run.


## 3. Inspect the selected preprocessing architecture

The goal is to make the comparison auditable.

Scenario A will be a fresh clone of the saved selected pipeline.  
Scenario B will clone the same pipeline and append one categorical transformer for `classification`.

No TF-IDF vocabulary, category vocabulary or fitted coefficient is reused from the final test-fitted model; the cloned pipelines are refitted only on the development training block.

In [3]:
selected_pipeline = bundle["pipeline"]

print("Pipeline steps:")
display(pd.Series(list(selected_pipeline.named_steps.keys()), name="step"))

if "preprocess" not in selected_pipeline.named_steps:
    raise KeyError("Saved pipeline has no `preprocess` step.")

pre = selected_pipeline.named_steps["preprocess"]

print("\nSaved preprocessor transformers:")
rows = []
for name, transformer, columns in pre.transformers:
    rows.append({
        "name": name,
        "transformer": type(transformer).__name__,
        "columns": columns,
    })
display(pd.DataFrame(rows))

Pipeline steps:


0    preprocess
1         model
Name: step, dtype: object


Saved preprocessor transformers:


,name,transformer,columns
0,subject,TfidfVectorizer,subject
1,origin,Pipeline,origin
2,categories,OneHotEncoder,"[category, type, notifying_country]"
3,hazards,Pipeline,hazards


## 4. Choose the same development split used for candidate comparison

The main rebuild keeps the final test period separate.  
Because `classification` is being tested **after** the final model was already selected, the default comparison here is performed on the development validation block rather than treating the final test set as a new model-selection set.

The code looks for `train` and either `valid` or `validation` in `temporal_blocks(df)`.

In [4]:
if "train" not in blocks:
    raise KeyError(f"`temporal_blocks` has no train block. Available: {list(blocks)}")

valid_key = "valid" if "valid" in blocks else ("validation" if "validation" in blocks else None)
if valid_key is None:
    raise KeyError(
        f"No validation block found. Available blocks: {list(blocks)}. "
        "Use the same development block name as Notebook 02."
    )

train_df = blocks["train"].copy()
valid_df = blocks[valid_key].copy()

def get_target(frame):
    if "target" in frame.columns:
        y = pd.to_numeric(frame["target"], errors="raise").astype(int)
    elif "risk_binary" in frame.columns:
        y = pd.to_numeric(frame["risk_binary"], errors="raise").astype(int)
    elif "risk_decision" in frame.columns:
        y = (
            frame["risk_decision"]
            .astype(str)
            .str.strip()
            .str.lower()
            .eq("serious")
            .astype(int)
        )
    else:
        raise KeyError("Could not resolve binary target from the prepared data.")
    return y

y_train = get_target(train_df)
y_valid = get_target(valid_df)

print(
    "Train:",
    len(train_df),
    train_df["date"].min() if "date" in train_df else "",
    "→",
    train_df["date"].max() if "date" in train_df else "",
)
print(
    "Validation:",
    len(valid_df),
    valid_df["date"].min() if "date" in valid_df else "",
    "→",
    valid_df["date"].max() if "date" in valid_df else "",
)

display(pd.DataFrame({
    "rows": [len(train_df), len(valid_df)],
    "serious_rate": [y_train.mean(), y_valid.mean()],
}, index=["train", "validation"]))

Train: 14668 2020-01-02 09:24:48 → 2023-05-17 18:41:05
Validation: 4748 2023-05-18 10:57:14 → 2024-04-18 17:56:13


,rows,serious_rate
train,14668,0.591764
validation,4748,0.483783


## 5. Build Scenario A from the exact saved pipeline

This is **not** a manually reconstructed `raw_lr_C2`.

It is a fresh sklearn clone of `bundle["pipeline"]`, so the model family, hyperparameters and all preprocessing choices are inherited from the selected architecture.

In [5]:
scenario_a = clone(selected_pipeline)

X_train_a = train_df[base_columns].copy()
X_valid_a = valid_df[base_columns].copy()

scenario_a.fit(X_train_a, y_train)

print("Scenario A fitted.")
print("Inputs:", base_columns)

Scenario A fitted.
Inputs: ['subject', 'origin', 'category', 'type', 'notifying_country', 'hazards']


## 6. Build Scenario B by adding only `classification`

To avoid inventing a new categorical preprocessing style, this notebook first tries to locate the transformer already used for `category` or `type` in the saved `ColumnTransformer` and clone that transformer for `classification`.

If no suitable existing categorical transformer can be identified, it falls back to `OneHotEncoder(handle_unknown="ignore")` and prints a warning.

This is the only architecture change between Scenario A and Scenario B.

In [6]:
def as_column_list(columns):
    if isinstance(columns, str):
        return [columns]
    try:
        return list(columns)
    except TypeError:
        return [columns]

def find_categorical_template(preprocessor):
    preferred_fields = ["category", "type", "notifying_country"]
    for field in preferred_fields:
        for name, transformer, columns in preprocessor.transformers:
            cols = as_column_list(columns)
            if field in cols and transformer not in ("drop", "passthrough"):
                return name, transformer, field
    return None, None, None

scenario_b = clone(selected_pipeline)
pre_b = scenario_b.named_steps["preprocess"]

template_name, template_transformer, template_field = find_categorical_template(pre_b)

if template_transformer is not None:
    classification_transformer = clone(template_transformer)
    print(
        f"Using cloned categorical transformer `{template_name}` "
        f"(template field: `{template_field}`) for `classification`."
    )
else:
    warnings.warn(
        "Could not identify an existing categorical transformer. "
        "Falling back to OneHotEncoder(handle_unknown='ignore')."
    )
    classification_transformer = OneHotEncoder(handle_unknown="ignore")

new_transformers = list(pre_b.transformers) + [
    ("classification_diagnostic", classification_transformer, ["classification"])
]

# Updating the transformer's public parameter keeps the rest of the saved
# ColumnTransformer configuration unchanged.
pre_b.set_params(transformers=new_transformers)

scenario_b_columns = base_columns + ["classification"]

X_train_b = train_df[scenario_b_columns].copy()
X_valid_b = valid_df[scenario_b_columns].copy()

scenario_b.fit(X_train_b, y_train)

print("Scenario B fitted.")
print("Inputs:", scenario_b_columns)

Using cloned categorical transformer `categories` (template field: `category`) for `classification`.
Scenario B fitted.
Inputs: ['subject', 'origin', 'category', 'type', 'notifying_country', 'hazards', 'classification']


## 7. Compare both scenarios at the same threshold

The comparison uses the same validation rows and threshold `0.5`.

Metrics:

- accuracy
- macro F1
- serious precision
- serious recall
- average precision
- ROC-AUC
- Brier score
- log loss

For Brier score and log loss, **lower is better**. For the other metrics, **higher is better**.

In [7]:
THRESHOLD = 0.5

def evaluate(name, model, X, y, threshold=0.5):
    p = model.predict_proba(X)[:, 1]
    pred = (p >= threshold).astype(int)
    return {
        "candidate": name,
        "accuracy": accuracy_score(y, pred),
        "macro_f1": f1_score(y, pred, average="macro"),
        "serious_precision": precision_score(y, pred, pos_label=1, zero_division=0),
        "serious_recall": recall_score(y, pred, pos_label=1, zero_division=0),
        "average_precision": average_precision_score(y, p),
        "roc_auc": roc_auc_score(y, p),
        "brier": brier_score_loss(y, p),
        "log_loss": log_loss(y, p),
        "threshold": threshold,
    }

results = pd.DataFrame([
    evaluate(
        "Scenario_A_selected_architecture",
        scenario_a,
        X_valid_a,
        y_valid,
        THRESHOLD,
    ),
    evaluate(
        "Scenario_B_plus_classification",
        scenario_b,
        X_valid_b,
        y_valid,
        THRESHOLD,
    ),
])

display(results)

,candidate,accuracy,macro_f1,serious_precision,serious_recall,average_precision,roc_auc,brier,log_loss,threshold
0,Scenario_A_selected_architecture,0.770008,0.769836,0.733437,0.824118,0.828613,0.846909,0.164797,0.515288,0.5
1,Scenario_B_plus_classification,0.842460,0.842444,0.809924,0.881149,0.894280,0.913537,0.118692,0.389451,0.5


## 8. Check Scenario A against the previously recorded selected-candidate result

The candidate table previously recorded for `raw_lr_C2` was approximately:

- Accuracy: **0.770008**
- Macro F1: **0.769836**
- Serious precision: **0.733437**
- Serious recall: **0.824118**
- Average precision: **0.828613**
- ROC-AUC: **0.846909**

If Scenario A is not close to those values, do not immediately interpret the classification delta.  
That would indicate that the development split or saved current architecture differs from the candidate table context.

This check is intentionally descriptive; it does not overwrite any result.

In [8]:
REFERENCE = {
    "accuracy": 0.770008,
    "macro_f1": 0.769836,
    "serious_precision": 0.733437,
    "serious_recall": 0.824118,
    "average_precision": 0.828613,
    "roc_auc": 0.846909,
}

a = results.set_index("candidate").loc["Scenario_A_selected_architecture"]

check = pd.DataFrame([
    {
        "metric": metric,
        "recorded_reference": ref,
        "scenario_A_rerun": float(a[metric]),
        "absolute_difference": abs(float(a[metric]) - ref),
    }
    for metric, ref in REFERENCE.items()
])

display(check)

max_diff = check["absolute_difference"].max()
if max_diff <= 0.01:
    print("PASS: Scenario A closely reproduces the recorded selected-candidate metrics.")
else:
    print(
        "NOTE: Scenario A does not closely reproduce every recorded candidate metric. "
        "Interpret the A/B comparison as a fresh controlled diagnostic using the current saved architecture."
    )

,metric,recorded_reference,scenario_A_rerun,absolute_difference
0,accuracy,0.770008,0.770008,4.245998e-07
1,macro_f1,0.769836,0.769836,1.206105e-07
2,serious_precision,0.733437,0.733437,3.475397e-07
3,serious_recall,0.824118,0.824118,4.153243e-07
4,average_precision,0.828613,0.828613,4.007846e-07
5,roc_auc,0.846909,0.846909,2.160193e-07


PASS: Scenario A closely reproduces the recorded selected-candidate metrics.


## 9. Quantify the effect of `classification`

The most important output of this notebook is **B − A**.

A positive delta in macro F1 / AP / ROC-AUC means `classification` adds predictive information under this development-period comparison.

That does **not** establish that the field is available early enough for the intended triage use case.

In [9]:
r = results.set_index("candidate")
a = r.loc["Scenario_A_selected_architecture"]
b = r.loc["Scenario_B_plus_classification"]

higher_better = [
    "accuracy",
    "macro_f1",
    "serious_precision",
    "serious_recall",
    "average_precision",
    "roc_auc",
]
lower_better = ["brier", "log_loss"]

delta_rows = []

for metric in higher_better:
    delta_rows.append({
        "metric": metric,
        "Scenario_A": a[metric],
        "Scenario_B": b[metric],
        "B_minus_A": b[metric] - a[metric],
        "interpretation": "positive = improvement",
    })

for metric in lower_better:
    delta_rows.append({
        "metric": metric,
        "Scenario_A": a[metric],
        "Scenario_B": b[metric],
        "B_minus_A": b[metric] - a[metric],
        "interpretation": "negative = improvement",
    })

delta = pd.DataFrame(delta_rows)
display(delta)

,metric,Scenario_A,Scenario_B,B_minus_A,interpretation
0,accuracy,0.770008,0.842460,0.072452,positive = improvement
1,macro_f1,0.769836,0.842444,0.072608,positive = improvement
2,serious_precision,0.733437,0.809924,0.076487,positive = improvement
3,serious_recall,0.824118,0.881149,0.057031,positive = improvement
4,average_precision,0.828613,0.894280,0.065667,positive = improvement
5,roc_auc,0.846909,0.913537,0.066628,positive = improvement
6,brier,0.164797,0.118692,-0.046105,negative = improvement
7,log_loss,0.515288,0.389451,-0.125837,negative = improvement


## 10. Inspect the transformed dimensions

This confirms that Scenario B differs by the extra encoded `classification` information rather than by accidentally dropping one of the six selected fields.

In [10]:
A_dim = scenario_a.named_steps["preprocess"].transform(X_valid_a.head(20)).shape[1]
B_dim = scenario_b.named_steps["preprocess"].transform(X_valid_b.head(20)).shape[1]

print("Scenario A transformed dimensions:", A_dim)
print("Scenario B transformed dimensions:", B_dim)
print("Additional transformed columns:", B_dim - A_dim)

Scenario A transformed dimensions: 8220
Scenario B transformed dimensions: 8225
Additional transformed columns: 5


## 11. Optional post-hoc check on the final test block — OFF by default

Notebook 03 already used the final test period once for the frozen final evaluation.

Because Scenario B was conceived **after** that evaluation, using the final test period now would be a post-hoc diagnostic, not a new untouched final test.

For that reason it is disabled by default.

If you later choose to run it for retrospective explanation, keep the result explicitly labeled **post-hoc** and do not use it to replace the selected model or claim a new final performance estimate.

In [11]:
RUN_POSTHOC_TEST = False

if RUN_POSTHOC_TEST:
    if "test" not in blocks:
        raise KeyError(f"No test block found. Available: {list(blocks)}")

    test_df = blocks["test"].copy()
    y_test = get_target(test_df)

    # Fit both diagnostic models on train + validation, mirroring the
    # development data available before the historical final test.
    development_df = pd.concat([train_df, valid_df], axis=0).sort_index()
    y_dev = get_target(development_df)

    post_a = clone(selected_pipeline)
    post_a.fit(development_df[base_columns], y_dev)

    post_b = clone(selected_pipeline)
    post_pre = post_b.named_steps["preprocess"]
    post_template_name, post_template_transformer, post_template_field = find_categorical_template(post_pre)

    if post_template_transformer is not None:
        post_cls_transformer = clone(post_template_transformer)
    else:
        post_cls_transformer = OneHotEncoder(handle_unknown="ignore")

    post_pre.set_params(
        transformers=list(post_pre.transformers) + [
            ("classification_diagnostic", post_cls_transformer, ["classification"])
        ]
    )

    post_b.fit(development_df[scenario_b_columns], y_dev)

    posthoc_results = pd.DataFrame([
        evaluate(
            "POSTHOC_A_selected_architecture",
            post_a,
            test_df[base_columns],
            y_test,
            THRESHOLD,
        ),
        evaluate(
            "POSTHOC_B_plus_classification",
            post_b,
            test_df[scenario_b_columns],
            y_test,
            THRESHOLD,
        ),
    ])

    display(posthoc_results)
else:
    print("Post-hoc final-test comparison not run.")

Post-hoc final-test comparison not run.


## 12. How to interpret the result

### If Scenario B improves substantially

Use this interpretation:

> Under the same current architecture and the same development-period evaluation, adding `classification` materially improved discrimination of the recorded serious label. This shows that `classification` carries strong information associated with `risk_decision`. Because its availability at the intended early prediction point is uncertain, it remains excluded from the primary model.

### If Scenario B improves only slightly

Then removing `classification` alone does not explain the difference between the earlier ~0.87 project and the corrected rebuild. Other changes — including chronology-aware evaluation, leakage-safe preprocessing, different feature representations and the removal of generated/late information — must also be considered.

### Important boundary

This notebook tests **predictive information**, not regulatory timing.

A high score with `classification` does not prove that the field is valid for a new-notification early-triage model.

## 13. Relationship to notebooks 03 and 04

- **03_Final_Evaluation_and_Model_Export.ipynb** remains the frozen final evaluation and export.
- **04_Interactive_Dashboard.ipynb** continues to read the original `final_bundle.joblib`.
- **05_Classification_LateInfo_Ablation.ipynb** is diagnostic only.
- Nothing in 05 overwrites `selected_spec.json`, `final_bundle.joblib`, `test_report.json`, or the app input contract.

The production/demo input contract therefore remains:

`subject`, `origin`, `category`, `type`, `notifying_country`, `hazards`